## What happens when an image enters YOLOv8-seg

---

**Step 1: The Backbone — "What's in this image?"**

The raw image (say 640×640 pixels) goes into a stack of convolutional layers. Each layer is basically a filter that slides across the image and learns to recognize patterns. Early layers learn simple things like edges and color gradients. Deeper layers combine those into more complex features like "this looks like a leg shape" or "this texture pattern is clothing."

The output of the backbone is not a decision — it's a set of **feature maps**. Think of these as a compressed, information-rich representation of the image. Instead of raw pixels, you now have a grid where each cell encodes "what kind of stuff is here" at multiple scales (fine detail and broad context).

Both the detection and segmentation heads read from these same feature maps. This is why they're connected — they're looking at the same processed information.

---

**Step 2: The Detection Head — "Where is the human?"**

The detection head takes the feature maps and predicts bounding boxes. For each potential object it outputs:

- **x, y, width, height** — a rectangle around the object
- **Class label** — "this is a person" (vs car, dog, etc.)
- **Confidence score** — a number from 0 to 1 representing how sure the model is that a real object of that class exists inside this box

That's it. A bounding box is literally just a rectangle. It says "somewhere inside this rectangle, there's a person, and I'm 0.91 confident." It tells you nothing about the shape of the person — it's always a rectangle regardless of whether the person is standing, crouching, or waving their arms.

---

**Step 3: The Segmentation Head — "Which exact pixels are the human?"**

This is where the silhouette comes from, and it works in two sub-steps:

**Sub-step A — Prototype generation.** A small network branch takes the feature maps and produces 32 **prototype masks**. Each prototype is a 160×160 grayscale image. These prototypes are abstract — individually they don't look like anything meaningful. One might highlight vertical edges, another might respond to the upper-left region, another to round shapes. They're a learned set of spatial building blocks.

**Sub-step B — Coefficient prediction.** For each detected person (from the detection head), the model also predicts 32 numbers — one coefficient per prototype. These coefficients say "use 0.8 of prototype #3, minus 0.2 of prototype #7, plus 0.5 of prototype #15..." and so on.

**Combining them:** The final mask is computed by:

```
mask = sigmoid(c1×proto1 + c2×proto2 + ... + c32×proto32)
```

The sigmoid squashes every pixel value into [0, 1]. Values near 1.0 mean "this pixel is the person." Values near 0.0 mean "this pixel is background." Values near 0.5 mean "the model has no idea."

The result is a pixel-level silhouette — the exact shape of the human, not just a rectangle.

---

## Bounding Box vs. Segmentation Mask — Visually

```
Bounding Box:                Segmentation Mask:
┌──────────────┐             
│              │                    ██
│   (person    │                  ██████
│    is in     │                   ████
│    here      │                  ██  ██
│  somewhere)  │                 ██    ██
│              │                ██      ██
└──────────────┘             
```

The bounding box is a rough "where." The segmentation mask is the precise "what shape."

---

## What your attack does

You're leaving the bounding box alone and corrupting the mask. After your attack:

```
Clean mask:                  Attacked mask:
                             
      ██                          █░░
    ██████                      ░██░█░░
     ████                        ░██░░█
    ██  ██                      █░  ░░
   ██    ██                    ░░░ ░░█░░
  ██      ██                  ░░░   ░░░░
```

The model still says "person detected at confidence 0.87" but the silhouette is noisy and wrong. The downstream gait recognition model that reads this silhouette now gets garbage data — it can't extract stride length, arm swing, or posture because the shape doesn't accurately represent the person's body anymore.

Does this make the full picture clearer? Want to go into how the loss function specifically targets the mask computation?

---

## Should you attack detection or segmentation?

Your instinct about detection is understandable — a dropping confidence score is a clean, simple number you can put on a poster. But let me lay out both options honestly:

**Attacking detection (make the model not see a human):**
- Easy to measure and present (one confidence number going down)
- Visually dramatic ("look, the bounding box disappeared")
- **But much harder to achieve.** Detection is the most robust part of YOLO. It's been trained on millions of images. Getting it to not see a person at all requires a very strong perturbation — probably stronger than what a realistic clothing patch can produce.

**Attacking segmentation (make the silhouette garbage):**
- More directly relevant to your gait recognition goal
- Easier to achieve because masks are more fragile than detection
- **But harder to present visually.** Showing a degraded mask is less immediately impressive than a vanishing bounding box

---

## The selling point of attacking segmentation

**You are not trying to hide from a camera. You are trying to be unrecognizable.**

Gait recognition's entire purpose is not just "is there a human" — that's trivial, any camera can see a person walking. The point is **"WHICH human is this?"** And that identification happens through the silhouette shape over time.

Think about it this way:

**The surveillance pipeline is:**
```
Camera → Detect person → Extract silhouette → Analyze gait → Match to database → "This is Baek"
```

**You don't need to break step 1. You need to break step 3.**

If the silhouette is corrupted, the gait analysis receives wrong body proportions, wrong limb boundaries, wrong motion patterns. It's like trying to identify someone's handwriting when you've smeared half the ink. The system says "I see a human walking" but it cannot say "this is the same human I saw yesterday."

---

## The real-world analogy

Imagine a surveillance system at an airport. It sees 500 people walking through. It doesn't need to figure out IF they're people — obviously they're people. The dangerous capability is that it can say "person #247 matches the walking pattern of this individual from our database."

Your patch doesn't make person #247 invisible. It makes person #247's gait signature **unreadable or inconsistent**. Every time the system sees them, it gets a different silhouette shape, so it can never build a stable profile.

### Make it impossible for surveillance systems to identify who you are based on how you walk, using a patterned piece of clothing.

---

## What confidence score can you see?

**Mask IoU (simplest, most presentable):**
Compare the clean silhouette to the attacked silhouette. IoU of 1.0 means they're identical. IoU of 0.0 means total destruction. You can show a graph of this number dropping as your patch gets better over training iterations. This is your "confidence score equivalent."

**Mean pixel confidence:**
Average the probability values across all pixels that should be the person. Clean might be 0.95. If your attack pushes it to 0.4, the model is barely sure those pixels are human at all.

**Pixel entropy:**
Measures how confused the model is per pixel. Clean silhouettes have low entropy (the model is certain). Attacked silhouettes have high entropy (the model is guessing).

---


# Gaitkeeper: Attack Strategy

## What YOLOv8-seg Gives Us

When a frame goes through YOLOv8-seg, the model produces **two separate outputs:**

**Output 1 — Detection (Bounding Box):** A rectangle around the person with a confidence score. "There's a person here, I'm 0.91 sure."

**Output 2 — Segmentation (Mask):** A pixel-level silhouette where every pixel has a probability between 0 and 1. Values near 1.0 = "this pixel is the person." Values near 0.0 = "this pixel is background."

Each output produces its own loss. **We only use the segmentation mask loss. We completely ignore the bounding box loss.** This is the core design decision — when we backpropagate gradients to update our patch, the gradients are shaped entirely by "how to make the mask worse," with zero signal from the detection output. This is what makes our attack a segmentation-specific attack rather than a general attack on the whole model.

---

## The Attack Loop

**Step 1 — Get ground truth.** Run a clean frame through YOLOv8-seg. Save the segmentation mask as M_clean. This is the correct silhouette we want to destroy.

**Step 2 — Apply patch.** Take the same frame. Paste the current patch onto the clothing region. Apply random transformations (rotation, brightness, perspective) to simulate real-world conditions. This produces the patched frame x'.

**Step 3 — Get attacked output.** Run x' through YOLOv8-seg. Extract ONLY the segmentation mask output: M_attack. Ignore the bounding box output entirely.

**Step 4 — Compute segmentation-only loss.** Compare M_attack to M_clean using two objectives:
- **Inversion loss:** Push person-pixels toward 0 (make the model think person is background)
- **Entropy loss:** Push person-pixels toward 0.5 (make the model maximally uncertain)

No bounding box loss. No detection confidence loss. No class prediction loss.

**Step 5 — Update patch.** Backpropagate the loss through the model to the patch. Update each pixel in the patch to make the mask worse next iteration. Clamp to printable color range [0, 255].

**Step 6 — Repeat** for 200-500 iterations across multiple frames and transformations until the mask is sufficiently degraded.

---

## What We Measure

| Metric | Clean | Target (Attacked) |
|---|---|---|
| Mask IoU (primary metric) | ~0.90 | < 0.40 |
| Mean pixel confidence (person region) | ~0.95 | < 0.50 |
| Mean pixel entropy (person region) | ~0.10 | > 0.50 |
| Detection confidence (tracked, NOT optimized) | ~0.90 | Side effect only |

---

## Why Segmentation Loss Only

The model gives us two losses. We use one. This is intentional:

- **Bounding box loss ignored** → Detection stays intact, model still sees a person
- **Segmentation mask loss used** → Silhouette gets corrupted, model can't trace the body shape
- **Result** → Person is detected but their silhouette is garbage → gait recognition receives unusable data → individual cannot be identified by their walk

# Gaitkeeper: Detailed Attack Strategy on YOLOv8-seg Segmentation Output

## Overview

We perform a white-box, gradient-based adversarial patch attack on YOLOv8-seg. The attack optimizes a printable pattern (the patch) to degrade the quality of the instance segmentation mask for detected persons. The key design decision is that **our loss function is computed exclusively on the segmentation mask output, not on the detection (bounding box) output.** This scopes the attack to corrupt the silhouette while leaving person detection largely intact — which is exactly what breaks downstream gait recognition.

---

## Step 1: Establish Ground Truth (Clean Pass)

### What happens

Take an unmodified frame from a walking video. Run it through YOLOv8-seg with no adversarial patch applied. Record everything the model outputs.

### What you extract

**Detection output:**
- Bounding box coordinates (x, y, w, h) for each detected person
- Class label (person) and detection confidence score
- These are saved but NOT used in the attack loss. They serve as reference only.

**Segmentation output:**
- 32 prototype masks from the proto-net (shape: 32 × 160 × 160)
- 32 mask coefficients per detected instance (shape: 1 × 32)
- The raw pre-sigmoid mask logits: `logits = coefficients @ prototypes` (shape: 160 × 160)
- The post-sigmoid mask probabilities: `mask = sigmoid(logits)` (shape: 160 × 160, values in [0, 1])
- The final binary mask after thresholding at 0.5 and upsampling to original resolution

**Ground truth mask (M_clean):**
The binary mask from this clean pass becomes the ground truth silhouette. Every pixel is labeled: 1 = person, 0 = background. This is what we want the attacked version to diverge from.

### Why this matters

We need M_clean as the reference point. The attack's success is measured entirely by how much the attacked mask M_attack differs from M_clean. Without this clean baseline, we have no way to quantify degradation.

---

## Step 2: Apply Patch and Get Attacked Output

### 2a: Define the patch

The adversarial patch δ is a learnable image tensor of shape (3, H_patch, W_patch) — three color channels, with height and width matching the clothing region. Every pixel in this tensor is a trainable parameter. Initialize it randomly (uniform noise) or from a solid color.

### 2b: Create the clothing bitmask

Before the attack loop begins, you need to know WHERE on the person's body to apply the patch. Run a separate clean segmentation pass (or use manual annotation) to create a **clothing region bitmask (B_clothing)**. This is a binary mask of just the torso area — not the full person silhouette, just the shirt/jacket region where the patch would physically exist.

This bitmask stays fixed throughout optimization. It defines the spatial constraint: the patch can only exist within this region.

### 2c: Apply the patch onto the frame

This is where the frame is modified. The patched frame x' is constructed as:

```
x' = x ⊙ (1 - B_clothing) + T(δ) ⊙ B_clothing
```

Where:
- `x` is the original clean frame
- `B_clothing` is the clothing region bitmask (1 where patch goes, 0 elsewhere)
- `T(δ)` is the patch after transformations (see 2d)
- `⊙` is element-wise multiplication

In plain English: keep every pixel outside the clothing region unchanged, replace every pixel inside the clothing region with the transformed patch.

### 2d: Apply Expectation over Transformation (EoT)

The patch is not applied under perfect digital conditions. Each time we apply it during training, we sample random transformations to simulate real-world variation:

**Geometric transforms:**
- Rotation: ±10-15° (the shirt won't be perfectly flat)
- Scale jitter: ±10% (camera distance varies)
- Perspective warp: slight trapezoid distortion (viewing angle)
- Translation: ±5 pixels (imperfect placement)

**Photometric transforms:**
- Brightness: ±20% (lighting conditions)
- Contrast: ±15%
- Saturation: ±10%
- Hue shift: ±5°

Each training iteration samples a DIFFERENT random combination of these transforms. This forces the patch to be robust across conditions, not overfit to one specific digital overlay.

### 2e: Forward pass the patched frame

Feed x' into YOLOv8-seg. The model processes it exactly as it would any normal image — backbone extracts features, detection head finds bounding boxes, segmentation head produces masks.

Extract the segmentation output for the detected person:
- Pre-sigmoid logits: `z_attack` (shape: 160 × 160)
- Post-sigmoid probabilities: `M_attack = sigmoid(z_attack)` (shape: 160 × 160, values in [0, 1])

**Critical: this forward pass must be done with gradients enabled.** In PyTorch, this means the patched image tensor x' must have `requires_grad` propagated through it, and the model must not be wrapped in `torch.no_grad()`. The model weights are frozen (we're not training the model), but the computation graph from input to output must be retained for backpropagation.

---

## Step 3: Compute the Segmentation-Only Loss

### Why this step is the core of the project

This is where the attack becomes specifically a segmentation attack rather than a general attack. We compute loss ONLY using the mask output. The detection head's bounding boxes and confidence scores are completely ignored in the loss computation. This means when we backpropagate, the gradients are shaped entirely by "how to make the mask worse," not "how to make detection worse."

### Loss function options

**Primary loss — Binary Cross Entropy against inverted ground truth:**

```
L_invert = BCE(M_attack, 1 - M_clean)
```

This tells the optimizer: "make every pixel that should be person look like background, and every pixel that should be background look like person." It directly trains the patch to flip the silhouette.

**Secondary loss — Entropy maximization over person region:**

```
L_entropy = -mean[ M_attack · log(M_attack) + (1 - M_attack) · log(1 - M_attack) ]
```

Computed only over pixels where M_clean = 1 (the person region). This pushes those pixels toward 0.5 — maximum uncertainty. The model can't decide if those pixels are person or background.

**Regularization — Total Variation on the patch:**

```
L_TV = Σ |δ[i,j] - δ[i+1,j]| + Σ |δ[i,j] - δ[i,j+1]|
```

This penalizes high-frequency noise in the patch. Without it, the optimizer might create a pattern with single-pixel variations that are effective digitally but impossible to print. TV regularization keeps the patch smooth enough to manufacture.

**Combined loss:**

```
L_total = λ1 · L_invert + λ2 · L_entropy + λ3 · L_TV
```

Where λ1, λ2, λ3 are hyperparameters controlling the balance. Start with λ1 = 1.0, λ2 = 0.5, λ3 = 0.01 and tune from there.

### What is NOT in the loss

- No bounding box regression loss
- No detection confidence loss  
- No class prediction loss

This is the engineering decision that makes this a segmentation-targeted attack. The gradients know nothing about detection.

---

## Step 4: Backpropagate and Update the Patch

### 4a: Compute gradients

Backpropagate L_total through the computation graph:

```
L_total → M_attack → segmentation head → backbone → input image x' → patch δ
```

This gives us ∂L_total/∂δ — the gradient of the loss with respect to every pixel in the patch. This gradient tells us: "for each pixel in the patch, which direction (brighter or darker, more red or more blue) would make the segmentation mask worse?"

### 4b: Update the patch using Projected Gradient Descent (PGD)

```
δ ← δ - α · sign(∂L_total/∂δ)
```

Where α is the step size (typically 1/255 to 4/255 per iteration). We use the sign of the gradient (not the raw gradient) for stability — this is the same principle as FGSM but applied iteratively.

After the update, **project the patch back to valid pixel range:**

```
δ ← clamp(δ, 0, 255)
```

This ensures the patch stays within printable color values.

### 4c: Why PGD over single-step FGSM

FGSM computes the gradient once and takes one big step. PGD takes many small steps, recomputing the gradient each time. PGD is strictly stronger because:
- The loss landscape is non-linear; one big step might overshoot
- Each iteration adapts to the new patch state
- The result is a more refined, effective pattern

### 4d: Batch and EoT averaging

In practice, each iteration doesn't use one frame with one transformation. It uses:
- A batch of B frames (e.g., B = 8) sampled from training videos
- N random EoT transformations per frame (e.g., N = 10)

The loss is averaged across all B × N = 80 forward passes before computing the gradient. This single gradient update accounts for multiple people, multiple poses, and multiple viewing conditions simultaneously.

```
L_total = (1 / B·N) Σ_b Σ_n L(frame_b, transform_n)
```

---

## Step 5: Iterate Until Convergence

Repeat steps 2-4 for 200-500 iterations. Track these metrics per iteration:

**Mask IoU** between M_clean and thresholded M_attack:
- Starting value: ~0.85-0.95 (patch hasn't learned anything yet)
- Target value: < 0.40 (silhouette is more wrong than right)

**Mean pixel probability** over person region:
- Starting value: ~0.90-0.98
- Target value: < 0.50 (model unsure if pixels are person)

**Mean pixel entropy** over person region:
- Starting value: ~0.05-0.15 (model very certain)
- Target value: > 0.50 (model maximally confused)

**Detection confidence** (tracked but not optimized):
- Starting value: ~0.85-0.95
- Likely ending value: ~0.60-0.85 (some degradation as side effect)

Plot all four over training iterations. You should see IoU dropping, entropy rising, and the patch visually evolving from random noise into a structured adversarial pattern.

---

## Step 6: Output

The final output is the optimized patch δ — an image file that can be:
- Digitally overlaid onto new test videos for synthetic evaluation
- Printed onto fabric or paper for physical-world evaluation

The patch is a fixed image. Once optimized, it doesn't change per person or per frame. The same pattern is applied to any subject, and the EoT training should make it effective across different people, poses, and conditions.

---

## Summary of What Makes This a Segmentation-Specific Attack

| Aspect | What we do | What a general attack would do |
|---|---|---|
| Loss function | Computed on mask output only | Computed on all outputs (box + class + mask) |
| Gradient signal | Shaped by segmentation pathway | Shaped by entire model |
| Goal | Corrupt silhouette shape | Prevent detection entirely |
| Detection impact | Incidental side effect | Primary objective |
| Gait recognition impact | Directly breaks silhouette analysis | Overkill — removes person entirely |
| Difficulty | Easier — masks are more fragile | Harder — detection is more robust |
| Presentation | "Silhouette accuracy: 93% → 41%" | "Detection confidence: 91% → 12%" |